## Fake and Real news dataset

In [1]:
import zipfile
import pandas as pd

# Extract ZIP
with zipfile.ZipFile(r"C:\Users\Rohit computer\OneDrive\Desktop\numpy.ds\archive (1).zip", 'r') as zip_ref:
    zip_ref.extractall("Fake_and_real_news_dataset")

# Load datasets
def load_data(filename):
    return pd.read_csv(filename)

fake = load_data("Fake_and_real_news_dataset/Fake.csv")
true = load_data("Fake_and_real_news_dataset/True.csv")


fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [2]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


## check missing values

In [3]:
true.isnull().sum()

title      0
text       0
subject    0
date       0
dtype: int64

In [4]:
fake.isnull().sum()

title      0
text       0
subject    0
date       0
dtype: int64

## Count duplicates in dataset

In [5]:
true.duplicated().sum()

np.int64(206)

In [6]:
fake.duplicated().sum()

np.int64(3)

In [7]:
true[true.duplicated()] 

,title,text,subject,date
445,Senate tax bill stalls on deficit-focused 'tri...,WASHINGTON (Reuters) - The U.S. Senate on Thur...,politicsNews,"November 30, 2017"
778,Trump warns 'rogue regime' North Korea of grav...,BEIJING (Reuters) - U.S. President Donald Trum...,politicsNews,"November 8, 2017"
892,"Republicans unveil tax cut bill, but the hard ...",WASHINGTON (Reuters) - U.S. House of Represent...,politicsNews,"November 2, 2017"
896,Trump taps Fed centrist Powell to lead U.S. ce...,WASHINGTON (Reuters) - President Donald Trump ...,politicsNews,"November 2, 2017"
974,"Two ex-Trump aides charged in Russia probe, th...",WASHINGTON (Reuters) - Federal investigators p...,politicsNews,"October 30, 2017"
...,...,...,...,...
21228,France unveils labor reforms in first step to ...,PARIS (Reuters) - French President Emmanuel Ma...,worldnews,"August 31, 2017"
21263,Guatemala top court sides with U.N. graft unit...,GUATEMALA CITY (Reuters) - Guatemala s top cou...,worldnews,"August 29, 2017"
21290,"Europeans, Africans agree renewed push to tack...",PARIS (Reuters) - Europe s big four continen...,worldnews,"August 28, 2017"
21353,Thailand's ousted PM Yingluck has fled abroad:...,BANGKOK (Reuters) - Ousted Thai prime minister...,worldnews,"August 25, 2017"


In [8]:
fake[fake.duplicated()]

,title,text,subject,date
9942,HILLARY TWEETS MESSAGE In Defense Of DACA…OOPS...,No time to waste we've got to fight with eve...,politics,"Sep 9, 2017"
11446,FORMER DEMOCRAT WARNS Young Americans: “Rioter...,"Who is silencing political speech, physically...",politics,"Mar 10, 2017"
14925,[VIDEO] #BlackLivesMatter Terrorists Storm Dar...,They were probably just looking for a safe sp...,politics,"Nov 16, 2015"


## Remove duplicate rows

In [9]:
fake = fake.drop_duplicates()
true = true.drop_duplicates()

In [10]:
fake[fake.duplicated()] ## check the rows have been remove or not

,title,text,subject,date


## preprocess Text

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

def preprocess_data(df):
    df = df.copy()  # ✅ Add this line to make a safe copy
    df['title'] = df['title'].str.lower()
    df['text'] = df['text'].str.lower() ##Converts all titles and texts to lowercase.
    return df

fake = preprocess_data(fake)
true = preprocess_data(true)

# Adding a new column 'label' to each DataFrame:
fake['label'] = 0   # Fake news = 0
true['label'] = 1   # True news = 1

data = pd.concat([fake, true]) ##Merges the two datasets (fake and true) into a single DataFrame called data.

# Combine title and text into one column
data['content'] = data['title'] + " " + data['text']
#EXPLAIN:-
# Concatenates the title and text into one string per row.
# Creates a new column called content that will be used for feature extraction.
# Example:
# If title = "Breaking News" and text = "India wins gold",
# then content = "breaking news india wins gold".

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X = vectorizer.fit_transform(data['content'])
#EXPLAIN:-
# TfidfVectorizer converts the text into numerical features for ML algorithms:
# stop_words='english' removes common words like the,is,and.
# max_features=5000 limits to the top 5000 words by importance.

# Labels
y = data['label']

## Train-test split

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Logistic Regression

In [13]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=200)
log_reg.fit(X_train, y_train)
log_reg_pred = log_reg.predict(X_test)

## Naive Bayes (MultinomialNB for text)

In [14]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)

## Random forest

In [16]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

## Evaluate and compare

In [17]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
     "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

# Train and evaluate
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"{name}\nAccuracy: {acc}\nPrecision: {prec}\nRecall: {rec}\nF1 Score: {f1}\n")

Logistic Regression
Accuracy: 0.9884761691653614
Precision: 0.9855415975349608
Recall: 0.99
F1 Score: 0.9877657679059271

Naive Bayes
Accuracy: 0.9346609979861267
Precision: 0.9346153846153846
Recall: 0.9257142857142857
F1 Score: 0.9301435406698565

Random Forest
Accuracy: 0.9978742447974939
Precision: 0.9978566325315551
Recall: 0.9976190476190476
F1 Score: 0.9977378259316585



## Save model and vectorizer

In [21]:
import pickle

# Example: If Logistic Regression performed best
best_model = LogisticRegression(max_iter=200)
best_model.fit(X_train, y_train)

with open("model.pkl", "wb") as f:
    pickle.dump(best_model, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("\n✅ Model and vectorizer saved successfully.")


✅ Model and vectorizer saved successfully.
